In [1]:
!pip install stable-baselines3[extra] datasets gymnasium torch torchvision


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Build the Model!

In [ ]:
import torch
import os
import csv
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecTransposeImage
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

from torchvision import models
from torch import nn

from datasets import load_dataset
import gymnasium as gym
from gymnasium import spaces



In [ ]:
class GeoGuessEnv(gym.Env):
    def __init__(self, dataset):
        super().__init__() # we inherit from parent class
        self.dataset = dataset
        self.index = 0

        # need to set the obersvation space to the size and shape of the images I am putting in!
        self.observation_space = spaces.Box(low=0, high=255, shape=(256, 384, 3), dtype=np.uint8)
        # action space is space of the ouput!
        # for simplicity sakkke, the prediction is between -1 and 1 and later scale that up!
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)

    def reset(self, seed=None, options=None): # need to define this method for each new start

        self.index = np.random.randint(len(self.dataset))
        obs = self.dataset[self.index]["image"]
        self.lat = float(self.dataset[self.index]["latitude"])
        self.lon = float(self.dataset[self.index]["longitude"])
        return np.array(obs), {}

    def step(self, action):
        # scale the prediction
        pred_lat = action[0] * 90
        pred_lon = action[1] * 180
        # find my distance from target
        distance_km = self.haversine(self.lon, self.lat, pred_lon, pred_lat)
        reward = -distance_km / 1000.0 # define reward here! Super nb!

        return np.array(self.dataset[self.index]["image"]), reward, True, False, {}

    def haversine(self, lon1, lat1, lon2, lat2):
        # as mentioned, simple distance calculator
        lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
        dlon = lon2 - lon1
        dlat = lat2 - lat1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        return 6371 * c


In [ ]:
"""
resnet feature extractor for RL stuff

loads a pretrained ResNet50 and chops off the last layer
just keeps the convolutional feature part. freezes the weights cos we not training it here.

used inside RL models to turn images into feature vectors

returns a 2048-dim feature vector per image
"""

class ResNetFeatureExtractor(BaseFeaturesExtractor):
    """
      uses a frozen resnet50 to extract features from image input

      params:
      - observation_space: spaces.Box
          box defining shape of input (like [3, 224, 224] for RGB images)

      forward:
      - input: torch tensor of shape [batch_size, 3, H, W]
      - scales to 0-1 by dividing by 255
      - returns: flattened tensor of shape [batch_size, 2048]
      """

    def __init__(self, observation_space: spaces.Box):
        super().__init__(observation_space, features_dim=2048)
        resnet = models.resnet50(pretrained=True)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        for param in self.backbone.parameters():
            param.requires_grad = False

    def forward(self, obs: torch.Tensor) -> torch.Tensor:

        obs = obs.float() / 255.0


        features = self.backbone(obs)
        return torch.flatten(features, 1)


class CustomCNNPolicy(ActorCriticPolicy):

    """
    custom ActorCriticPolicy that plugs in a ResNet50 as feature extractor

    no extra args needed – just wraps the base policy and swaps the feature extractor

    args:
    - *args, **kwargs: whatever you’d normally pass to the sb3 policy

    sets:
    - features_extractor_class = ResNetFeatureExtractor
    - features_extractor_kwargs = {} (empty for now, but can be added later)
    """

    def __init__(self, *args, **kwargs):
        super().__init__(
            *args,
            **kwargs,
            features_extractor_class=ResNetFeatureExtractor,
            features_extractor_kwargs={}
        )


In [ ]:
import csv
from stable_baselines3.common.callbacks import BaseCallback

class CSVLoggerCallback(BaseCallback):
    """
    Very not critical, but used for logging!

    Just so I can see what's happening throughout!
    """

    def __init__(self, csv_path, verbose=0):
        super().__init__(verbose)
        self.csv_path = csv_path
        self.headers = [
            "timesteps", "iterations", "reward_mean", "value_loss",
            "policy_loss", "entropy", "clip_frac", "kl"
        ]
        with open(self.csv_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(self.headers)

    def _on_rollout_end(self) -> None:
        logs = self.logger.name_to_value

        # Print reward (episode mean)
        ep_rew = logs.get("rollout/ep_rew_mean", None)
        if ep_rew is not None:
            print(f"[Rollout] Avg Reward: {ep_rew:.2f}")

        # Existing logging to CSV
        with open(self.csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                logs.get("time/total_timesteps", "NA"),
                logs.get("time/iterations", "NA"),
                ep_rew or "",
                logs.get("train/value_loss", ""),
                logs.get("train/policy_gradient_loss", ""),
                logs.get("train/entropy_loss", ""),
                logs.get("train/clip_fraction", ""),
                logs.get("train/approx_kl", "")
            ])

    def _on_step(self) -> bool:
        # Really do not need this function, but the package (Stable Baselines3) expects it.
        # Does not run without, so added it.
        return True


## Import models and get set up!


In [ ]:
def make_env(dataset):
    def _init():
        return GeoGuessEnv(dataset)
    return _init


from datasets import load_from_disk

dataset_dict = load_from_disk("processed_streetview_rect")
train_ds = dataset_dict["train"]
test_ds = dataset_dict["test"]


train_env = DummyVecEnv([make_env(train_ds)])


eval_env = DummyVecEnv([make_env(test_ds)])


train_env = DummyVecEnv([make_env(train_ds)])
eval_env = DummyVecEnv([make_env(test_ds)])


eval_callback = EvalCallback(eval_env, eval_freq=10000,
                             best_model_save_path="./checkpoints",
                             log_path="./logs", verbose=1)

csv_logger = CSVLoggerCallback("training_progress.csv")



# Hyper Parameter tuning
- The goal is to use random search to pick hyper params in some range, train a model for less than normal epochs and see what is working and not.
- Then I use the best hyper params

In [ ]:
import random
from stable_baselines3.common.callbacks import CallbackList


param_grid = {
    "learning_rate": [1e-5, 1e-4, 3e-4, 1e-3],
    "n_steps": [256, 512, 1024],
    "batch_size": [256, 512, 1024],
    "gamma": [0.95, 0.98, 0.99],
    "clip_range": [0.1, 0.2, 0.3]
}


n_trials = 10
results = []

for i in range(n_trials):
    config = {k: random.choice(v) for k, v in param_grid.items()}


    print(f"doing trial: {i+1} on confif: {config}")

    model = PPO(CustomCNNPolicy, train_env,
                **config,
                verbose=1)
    callback = CallbackList([csv_logger, eval_callback])

    model.learn(total_timesteps=25000, callback=callback)  # shorter runs than normal

    # Evaluate or log metrics yourself
    mean_reward = eval_callback.last_mean_reward if hasattr(eval_callback, "last_mean_reward") else None
    print("meanie")
    print(mean_reward)
    results.append((config, mean_reward))

# Sort and find the best
results.sort(key=lambda x: x[1], reverse=True)
print("Best config:", results[0][0])
print("Best reward:", results[0][1])


doing trial: 1 on confif: {'learning_rate': 0.001, 'n_steps': 256, 'batch_size': 1024, 'gamma': 0.99, 'clip_range': 0.3}
Using cuda device
Wrapping the env in a VecTransposeImage.


/usr/local/lib/python3.11/dist-packages/stable_baselines3/ppo/ppo.py:155: UserWarning: You have specified a mini-batch size of 1024, but because the `RolloutBuffer` is of size `n_steps * n_envs = 256`, after every 0 untruncated mini-batches, there will be a truncated mini-batch of size 256
We recommend using a `batch_size` that is a factor of `n_steps * n_envs`.
Info: (n_steps=256 and n_envs=1)
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7d05f01e0ad0> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x7d095922b510>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


----------------------------
| time/              |     |
|    fps             | 37  |
|    iterations      | 1   |
|    time_elapsed    | 6   |
|    total_timesteps | 256 |
----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 30          |
|    iterations           | 2           |
|    time_elapsed         | 16          |
|    total_timesteps      | 512         |
| train/                  |             |
|    approx_kl            | 0.048640974 |
|    clip_fraction        | 0.0848      |
|    clip_range           | 0.3         |
|    entropy_loss         | -2.84       |
|    explained_variance   | 0           |
|    learning_rate        | 0.001       |
|    loss                 | 20          |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0477     |
|    std                  | 0.996       |
|    value_loss           | 64          |
-----------------------------------------

/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


-----------------------------
| time/              |      |
|    fps             | 25   |
|    iterations      | 18   |
|    time_elapsed    | 179  |
|    total_timesteps | 4608 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 25          |
|    iterations           | 19          |
|    time_elapsed         | 189         |
|    total_timesteps      | 4864        |
| train/                  |             |
|    approx_kl            | 0.032609418 |
|    clip_fraction        | 0.105       |
|    clip_range           | 0.3         |
|    entropy_loss         | -2.76       |
|    explained_variance   | 0.0118      |
|    learning_rate        | 0.001       |
|    loss                 | 5.33        |
|    n_updates            | 180         |
|    policy_gradient_loss | -0.0061     |
|    std                  | 0.966       |
|    value_loss           | 10.7        |
----------------------------------

...

----------------------------
| time/              |     |
|    fps             | 38  |
|    iterations      | 1   |
|    time_elapsed    | 13  |
|    total_timesteps | 512 |
----------------------------
-------------------------------------------
| time/                   |               |
|    fps                  | 30            |
|    iterations           | 2             |
|    time_elapsed         | 33            |
|    total_timesteps      | 1024          |
| train/                  |               |
|    approx_kl            | 1.9628671e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.1           |
|    entropy_loss         | -2.84         |
|    explained_variance   | 0             |
|    learning_rate        | 1e-05         |
|    loss                 | 58.9          |
|    n_updates            | 10            |
|    policy_gradient_loss | 0.000294      |
|    std                  | 1             |
|    value_loss           | 119           |
-----

/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7d0648a58b10> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x7d095922b510>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


----------------------------
| time/              |     |
|    fps             | 37  |
|    iterations      | 1   |
|    time_elapsed    | 13  |
|    total_timesteps | 512 |
----------------------------
-------------------------------------------
| time/                   |               |
|    fps                  | 29            |
|    iterations           | 2             |
|    time_elapsed         | 34            |
|    total_timesteps      | 1024          |
| train/                  |               |
|    approx_kl            | 0.00012770772 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -2.84         |
|    explained_variance   | -1.19e-07     |
|    learning_rate        | 0.0001        |
|    loss                 | 48            |
|    n_updates            | 10            |
|    policy_gradient_loss | -0.00315      |
|    std                  | 0.999         |
|    value_loss           | 110           |
-----

/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7d05cc233510> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x7d095922b510>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


----------------------------
| time/              |     |
|    fps             | 36  |
|    iterations      | 1   |
|    time_elapsed    | 13  |
|    total_timesteps | 512 |
----------------------------
-------------------------------------------
| time/                   |               |
|    fps                  | 29            |
|    iterations           | 2             |
|    time_elapsed         | 34            |
|    total_timesteps      | 1024          |
| train/                  |               |
|    approx_kl            | 0.00038176368 |
|    clip_fraction        | 0             |
|    clip_range           | 0.3           |
|    entropy_loss         | -2.84         |
|    explained_variance   | 0             |
|    learning_rate        | 0.0001        |
|    loss                 | 49            |
|    n_updates            | 10            |
|    policy_gradient_loss | -0.00972      |
|    std                  | 1             |
|    value_loss           | 112           |
-----

# Final Train
- I use best hyper params and do a final train!

In [ ]:
best_config = results[0][0]

model = PPO(CustomCNNPolicy, train_env,
            **best_config,
            verbose=1)





model.learn(total_timesteps=250_000, callback=[eval_callback, csv_logger])
model.save("ppo_geoguess")


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x7830aa17a110> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x7830a82403d0>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


[DEBUG] Input shape to ResNetFeatureExtractor: torch.Size([1, 3, 256, 384])
[DEBUG] After permute: torch.Size([1, 3, 256, 384])
[DEBUG] Input shape to ResNetFeatureExtractor: torch.Size([1, 3, 256, 384])
[DEBUG] After permute: torch.Size([1, 3, 256, 384])
[DEBUG] Input shape to ResNetFeatureExtractor: torch.Size([1, 3, 256, 384])
[DEBUG] After permute: torch.Size([1, 3, 256, 384])
[DEBUG] Input shape to ResNetFeatureExtractor: torch.Size([1, 3, 256, 384])
[DEBUG] After permute: torch.Size([1, 3, 256, 384])
[DEBUG] Input shape to ResNetFeatureExtractor: torch.Size([1, 3, 256, 384])
[DEBUG] After permute: torch.Size([1, 3, 256, 384])
[DEBUG] Input shape to ResNetFeatureExtractor: torch.Size([1, 3, 256, 384])
[DEBUG] After permute: torch.Size([1, 3, 256, 384])
[DEBUG] Input shape to ResNetFeatureExtractor: torch.Size([1, 3, 256, 384])
[DEBUG] After permute: torch.Size([1, 3, 256, 384])
[DEBUG] Input shape to ResNetFeatureExtractor: torch.Size([1, 3, 256, 384])
[DEBUG] After permute: torch

KeyboardInterrupt: 